In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

!pwd
import os
os.chdir('/content/drive/My Drive/Colab Notebooks')
!pwd
# Get the current working directory
current_directory = os.getcwd()

# Specify the file name
file_name = "mastershifted.txt"

# Construct the full path to the file
file_path = os.path.join(current_directory, file_name)

# Open the file
try:
    with open(file_path, "r") as f:
        # Do something with the file (e.g., print its contents)
        print("Loaded successfully")
except FileNotFoundError:
    print(f"The file '{file_name}' was not found in the current directory.")
except Exception as e:
    print(f"An error occurred: {e}")

/content
/content/drive/My Drive/Colab Notebooks
Loaded successfully


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

colnames=( 'Bx', 'By', 'Bz', 'Vx', 'Density', 'Labels')
df = pd.read_csv('mastershifted.txt', sep='\s+', names=colnames)
dfc=df.drop(['Labels'], axis=1)

n_timesteps = 120

Xcols=[x for x in dfc.columns if x!= 'Labels']
n_features=len(Xcols)


scaler = MinMaxScaler()
X = df[Xcols].values
y=df['Labels']
targtrain=np.empty((53404,))
targtrain[::2]=0 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]
targtrain[1::2]=1 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]

targtest=np.empty((11444,))
targtest[::2]=0 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]
targtest[1::2]=1 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]

targvalid=np.empty((11444,))
targvalid[::2]=0 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]
targvalid[1::2]=1 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]

X_train = X[0:6408480]
y_train = targtrain#y[0:4700760]
X_test =  X[6408480:7781760]
y_test =  targtest#y[4700760:5708040]
X_val =   X[7781760:9155040]
y_val =   targvalid#y[5708040:6715320]



X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_val=scaler.transform(X_val)

n_samples = int(X_train.shape[0]/120)
X_tr = np.resize(X_train, (n_samples, 600))
X_te = np.resize(X_test, (11444, 600))
X_va = np.resize(X_val, (11444, 600))

dtrain = xgb.DMatrix(X_tr, label=y_train)
dtest = xgb.DMatrix(X_te, label=y_test)
dvalid = xgb.DMatrix(X_va, label=y_val)


# _____________________________________________________________________


In [ ]:
# Set up parameters for binary classification with XGBoost
params = {
    'objective': 'binary:logistic',  # Binary classification task
    'eval_metric': ['logloss'],        # Log-loss metric for binary classification
    #'gamma':5,
    'max_depth': 5,
    'learning_rate': 0.0001,
    'lambda': 100,
    'alpha': 100,
}
evals = [(dtrain, 'train'), (dtest, 'test'), (dvalid, 'validation')]
evals_result = {}

# Train the XGBoost model
xgbmodel = xgb.train(params, dtrain, num_boost_round=5000,evals=evals,early_stopping_rounds=100, evals_result=evals_result,  verbose_eval=True)



In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.linear_model import LogisticRegression

# Get the raw prediction scores (logits) from XGBoost
#raw_preds = xgbmodel.predict(dvalid, output_margin=True)  # Get the raw logits, not probabilities
raw_pred_train = xgbmodel.predict(dtrain, output_margin=True)
raw_pred_test = xgbmodel.predict(dtest, output_margin=True)
raw_pred_val = xgbmodel.predict(dvalid, output_margin=True)


# Apply Platt Scaling via sklearn's CalibratedClassifierCV
# First, you need to fit a logistic regression model to these raw predictions
log_reg = LogisticRegression()
log_reg.fit(raw_pred_train.reshape(-1, 1), y_train)
log_reg.fit(raw_pred_test.reshape(-1, 1), y_test)
log_reg.fit(raw_pred_val.reshape(-1, 1), y_val)

# Then, we calibrate the predictions
calibrated_pred_train = log_reg.predict_proba(raw_pred_train.reshape(-1, 1))[:, 1]
calibrated_pred_test = log_reg.predict_proba(raw_pred_test.reshape(-1, 1))[:, 1]
calibrated_pred_val = log_reg.predict_proba(raw_pred_val.reshape(-1, 1))[:, 1]
